<a href="https://colab.research.google.com/github/laitooo/anime-character-recognizer/blob/main/ai/colab_anime_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anime Character Classifier (Notebook-Only)

This notebook trains and evaluates an anime character classifier from face images using a dataset downloaded with `kagglehub`.

In [1]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "tensorflow",
    "numpy",
    "Pillow",
    "scikit-learn",
    "kagglehub",
    "opencv-python-headless",
])

import json
import random
from pathlib import Path

import cv2
import kagglehub
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split

## Runtime Setup
Install dependencies and import all required libraries. Run this once for each fresh Colab runtime.

In [2]:
SEED = 42
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 40

DATASET_SLUG = "anuragraj03/anime-face-dataset"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

ARTIFACTS_DIR = Path("artifacts")
MODEL_DIR = ARTIFACTS_DIR / "model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Constants configured.")

Constants configured.


## Shared Helpers
Define reusable helper functions for dataset handling, preprocessing, and model construction.

In [3]:
def list_class_dirs(source_dir: Path):
    class_dirs = [d for d in source_dir.iterdir() if d.is_dir() and not d.name.startswith(".")]
    return sorted(class_dirs, key=lambda p: p.name.lower())


def list_images(class_dir: Path):
    return sorted([f for f in class_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])


def decode_and_resize(image_path, label):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


def make_dataset(records, shuffle=True):
    paths = [str(path) for path, _ in records]
    labels = [label for _, label in records]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle and records:
        ds = ds.shuffle(buffer_size=len(records), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def build_augmentation():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomTranslation(0.08, 0.08),
        tf.keras.layers.RandomContrast(0.15),
    ], name="augmentation")


def conv_block(filters: int):
    return [
        tf.keras.layers.Conv2D(filters, (3, 3), padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2D(filters, (3, 3), padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D((2, 2)),
    ]


def build_model(num_classes: int):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        build_augmentation(),
        *conv_block(32),
        *conv_block(64),
        *conv_block(128),
        tf.keras.layers.SpatialDropout2D(0.1),
        *conv_block(256),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ], name="anime_cnn")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


print("Shared helper functions ready.")

Shared helper functions ready.


## Dataset Check
Download and validate the dataset directory, then show class counts and image distribution.

In [4]:
dataset_path = Path(kagglehub.dataset_download(DATASET_SLUG))
SOURCE_DIR = dataset_path

class_dirs = list_class_dirs(SOURCE_DIR)
if not class_dirs:
    nested_dirs = [d for d in SOURCE_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")]
    if len(nested_dirs) == 1:
        SOURCE_DIR = nested_dirs[0]
        class_dirs = list_class_dirs(SOURCE_DIR)

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")
if not class_dirs:
    raise ValueError(f"No class folders found in {SOURCE_DIR}")

summary = {d.name: len(list_images(d)) for d in class_dirs}
print(f"Source directory: {SOURCE_DIR.resolve()}")
print(f"Number of classes: {len(summary)}")
print(f"Total images: {sum(summary.values())}")
print("Images per class:")
for class_name, count in summary.items():
    print(f"  - {class_name}: {count}")

100%|██████████| 251M/251M [00:02<00:00, 90.3MB/s]

Extracting files...


Source directory: /root/.cache/kagglehub/datasets/anuragraj03/anime-face-dataset/versions/1
Number of classes: 31
Total images: 3720
Images per class:
  - Android 18: 120
  - Armin Arlert: 120
  - Boa Hancock: 120
  - Eren Yeager: 120
  - Goku: 120
  - Hinata Hyuga: 120
  - Historia Reiss: 120
  - Inosuke Hashibira: 120
  - Kakashi Hatake: 120
  - Katsuhiko Teshigawara: 120
  - Levi Ackerman: 120
  - Megumi Fushiguro: 120
  - Mikasa Ackerman: 120
  - Monkey D. Luffy: 120
  - Nami: 120
  - Naruto Uzumaki: 120
  - Nezuko Kamado: 120
  - Nico Robin: 120
  - Roronoa Zoro: 120
  - Saitama: 120
  - Sakura Haruno: 120
  - Sanji: 120
  - Sasha Blouse: 120
  - Sasuke Uchiha: 120
  - Satoru Gojo: 120
  - Spike Spiegel: 120
  - Taki Tachibana: 120
  - Tsunade: 120
  - Vegeta: 120
  - Yuji Itadori: 120
  - Zenitsu Agatsuma: 120


## Train/Validation/Test Split
Split image paths into in-memory train/validation/test records and print split sizes.

In [5]:
class_names = [d.name for d in class_dirs]
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

train_records, val_records, test_records = [], [], []
split_stats = {"train": {}, "val": {}, "test": {}}

for class_dir in class_dirs:
    images = list_images(class_dir)
    if len(images) < 3:
        raise ValueError(f"Class '{class_dir.name}' has less than 3 images")

    train_files, remaining = train_test_split(
        images,
        test_size=0.30,
        random_state=SEED,
        shuffle=True,
    )
    val_files, test_files = train_test_split(
        remaining,
        test_size=0.50,
        random_state=SEED,
        shuffle=True,
    )

    label = class_to_idx[class_dir.name]
    train_records.extend((path, label) for path in train_files)
    val_records.extend((path, label) for path in val_files)
    test_records.extend((path, label) for path in test_files)

    split_stats["train"][class_dir.name] = len(train_files)
    split_stats["val"][class_dir.name] = len(val_files)
    split_stats["test"][class_dir.name] = len(test_files)

print("Dataset split completed (in memory).")
for split_name, records in (("train", train_records), ("val", val_records), ("test", test_records)):
    print(f"{split_name}: {len(records)} images")

Dataset split completed (in memory).
train: 2604 images
val: 558 images
test: 558 images


## Model Definition
Build the CNN classifier and save the model summary artifact.

In [6]:
num_classes = len(class_dirs)
model = build_model(num_classes)
model.summary()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
summary_lines = []
model.summary(print_fn=lambda x: summary_lines.append(x))
(ARTIFACTS_DIR / "model_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Model summary saved to: {(ARTIFACTS_DIR / 'model_summary.txt').resolve()}")

Model: "anime_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             

 Total params: 1,248,895 (4.76 MB)

 Trainable params: 1,246,975 (4.76 MB)

 Non-trainable params: 1,920 (7.50 KB)

Model summary saved to: /content/artifacts/model_summary.txt


## CNN Structure
Write a plain-text description of the CNN architecture to an artifact file.

In [7]:
cnn_structure = (
    "CNN structure used in this project:\n\n"
    "Input: 128x128x3 RGB image\n"
    "-> Input normalization in tf.data pipeline (pixel / 255)\n"
    "-> Data augmentation (training only): RandomFlip(horizontal), RandomRotation(0.08),\n"
    "   RandomZoom(0.1), RandomTranslation(0.08), RandomContrast(0.15)\n"
    "-> Conv block 32:  [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)\n"
    "-> Conv block 64:  [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)\n"
    "-> Conv block 128: [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)\n"
    "-> SpatialDropout2D(0.1)\n"
    "-> Conv block 256: [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)\n"
    "-> GlobalAveragePooling2D\n"
    "-> Dropout(0.4)\n"
    "-> Dense(256, relu)\n"
    "-> Dropout(0.4)\n"
    "-> Dense(num_classes, softmax)\n"
)

(ARTIFACTS_DIR / "cnn_structure.txt").write_text(cnn_structure, encoding="utf-8")
print(cnn_structure)
print(f"CNN structure saved to: {(ARTIFACTS_DIR / 'cnn_structure.txt').resolve()}")

CNN structure used in this project:

Input: 128x128x3 RGB image
-> Input normalization in tf.data pipeline (pixel / 255)
-> Data augmentation (training only): RandomFlip(horizontal), RandomRotation(0.08),
   RandomZoom(0.1), RandomTranslation(0.08), RandomContrast(0.15)
-> Conv block 32:  [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)
-> Conv block 64:  [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)
-> Conv block 128: [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)
-> SpatialDropout2D(0.1)
-> Conv block 256: [Conv2D(3x3, same) -> BatchNorm -> ReLU] x2 -> MaxPooling2D(2x2)
-> GlobalAveragePooling2D
-> Dropout(0.4)
-> Dense(256, relu)
-> Dropout(0.4)
-> Dense(num_classes, softmax)

CNN structure saved to: /content/artifacts/cnn_structure.txt


## Training
Train the model on the training split, validate each epoch, and save best/final checkpoints.

In [8]:
train_ds = make_dataset(train_records, shuffle=True)
val_ds = make_dataset(val_records, shuffle=False)

(MODEL_DIR / "class_names.json").write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

model = build_model(num_classes=len(class_names))
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_model.keras"),
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
model.save(MODEL_DIR / "final_model.keras")
print(f"Training complete. Final model saved to: {(MODEL_DIR / 'final_model.keras').resolve()}")
print(f"Best model saved to: {(MODEL_DIR / 'best_model.keras').resolve()}")
print("Final epoch metrics:", {k: float(v[-1]) for k, v in history.history.items()})

Epoch 1/40
82/82 ━━━━━━━━━━━━━━━━━━━━ 299s 4s/step - accuracy: 0.1129 - loss: 3.1965 - val_accuracy: 0.0376 - val_loss: 3.7469 - learning_rate: 0.0010
Epoch 2/40
82/82 ━━━━━━━━━━━━━━━━━━━━ 330s 4s/step - accuracy: 0.2162 - loss: 2.6426 - val_accuracy: 0.0466 - val_loss: 6.0556 - learning_rate: 0.0010
Epoch 3/40
82/82 ━━━━━━━━━━━━━━━━━━━━ 316s 4s/step - accuracy: 0.2719 - loss: 2.4469 - val_accuracy: 0.0538 - val_loss: 5.9678 - learning_rate: 0.0010
Epoch 4/40
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3487 - loss: 2.2415
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
82/82 ━━━━━━━━━━━━━━━━━━━━ 320s 4s/step - accuracy: 0.3514 - loss: 2.2256 - val_accuracy: 0.1326 - val_loss: 4.0761 - learning_rate: 0.0010
Epoch 5/40
82/82 ━━━━━━━━━━━━━━━━━━━━ 297s 4s/step - accuracy: 0.4109 - loss: 1.9869 - val_accuracy: 0.1685 - val_loss: 3.5033 - learning_rate: 5.0000e-04
Epoch 6/40
82/82 ━━━━━━━━━━━━━━━━━━━━ 322s 4s/step - accuracy: 0.4382 - loss: 1.8636 - val_ac

## Evaluation
Evaluate the trained model on the test split and print loss/accuracy metrics.

In [9]:
test_ds = make_dataset(test_records, shuffle=False)

best_model_path = MODEL_DIR / "best_model.keras"
model_path = best_model_path if best_model_path.exists() else (MODEL_DIR / "final_model.keras")
trained_model = tf.keras.models.load_model(model_path)

loss, accuracy = trained_model.evaluate(test_ds, verbose=1)
print(f"Loaded model: {model_path.resolve()}")
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

18/18 ━━━━━━━━━━━━━━━━━━━━ 15s 783ms/step - accuracy: 0.8172 - loss: 0.5899
Loaded model: /content/artifacts/model/best_model.keras
Classes (31): ['Android 18', 'Armin Arlert', 'Boa Hancock', 'Eren Yeager', 'Goku', 'Hinata Hyuga', 'Historia Reiss', 'Inosuke Hashibira', 'Kakashi Hatake', 'Katsuhiko Teshigawara', 'Levi Ackerman', 'Megumi Fushiguro', 'Mikasa Ackerman', 'Monkey D. Luffy', 'Nami', 'Naruto Uzumaki', 'Nezuko Kamado', 'Nico Robin', 'Roronoa Zoro', 'Saitama', 'Sakura Haruno', 'Sanji', 'Sasha Blouse', 'Sasuke Uchiha', 'Satoru Gojo', 'Spike Spiegel', 'Taki Tachibana', 'Tsunade', 'Vegeta', 'Yuji Itadori', 'Zenitsu Agatsuma']
Test loss: 0.5899
Test accuracy: 0.8172


## Single Image Prediction
Predict a character for one image, using face detection/cropping before classification.

In [10]:
try:
    from google.colab import files
    uploaded = files.upload()
    uploaded_path = list(uploaded.keys())[0]
except Exception:
    uploaded_path = input("Enter image path to test: ").strip()

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if face_cascade.empty():
    raise RuntimeError("Failed to load Haar face detector.")

original_img = Image.open(uploaded_path).convert("RGB")
rgb = np.array(original_img)
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48))

if len(faces) == 0:
    face_img = original_img
    face_bbox = None
else:
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
    pad = int(0.2 * max(w, h))
    x0 = max(0, x - pad)
    y0 = max(0, y - pad)
    x1 = min(rgb.shape[1], x + w + pad)
    y1 = min(rgb.shape[0], y + h + pad)
    face_img = Image.fromarray(rgb[y0:y1, x0:x1])
    face_bbox = (x0, y0, x1, y1)

model_img = face_img.resize((IMG_SIZE, IMG_SIZE))
arr = np.expand_dims(np.array(model_img, dtype=np.float32) / 255.0, axis=0)

class_names = json.loads((MODEL_DIR / "class_names.json").read_text(encoding="utf-8"))
probs = trained_model.predict(arr, verbose=0)[0]
pred_idx = int(np.argmax(probs))
top_indices = np.argsort(probs)[::-1][: min(3, len(probs))]

print(f"Image: {uploaded_path}")
if face_bbox is None:
    print("Face detection: no face found, used full image.")
else:
    print(f"Face detection: used crop bbox={face_bbox}")
print(f"Predicted character: {class_names[pred_idx]}")
print(f"Confidence: {float(probs[pred_idx]):.6f}")
print("Top predictions:")
for idx in top_indices:
    print(f"  - {class_names[int(idx)]}: {float(probs[int(idx)]):.6f}")

Saving eren.jpeg to eren.jpeg


AttributeError: module 'cv2' has no attribute 'CascadeClassifier'

## Save Weights
Save model weights for reuse in future sessions.

In [11]:
weights_path = ARTIFACTS_DIR / "model_weights.weights.h5"
weights_path.parent.mkdir(parents=True, exist_ok=True)
trained_model.save_weights(weights_path)
print(f"Weights saved to: {weights_path.resolve()}")

Weights saved to: /content/artifacts/model_weights.weights.h5
